# Policies

> Every policy type a `World` can run, with the interface of stable-worldmodel.

In [ ]:
#| default_exp policy

In [ ]:
#| hide
from nbdev.showdoc import *

A policy receives the world's stacked **info dict** and returns one action per env and agent.

Every info value has a leading ``(num_envs, 1)`` (env, time), as in stable-worldmodel; per-agent
values add the agent axis after it, ``(num_envs, 1, num_agents, ...)``. Actions are returned
as an array of shape ``(num_envs, num_agents)``.

| policy | acts from |
|---|---|
| `RandomPolicy` | uniform samples of the action space |
| `ExpertPolicy` | the true state of each env (scripted, e.g. `GoToGoalPolicy`) |
| `FeedForwardPolicy` | one forward pass of a torch model (behaviour cloning, MARL actors) |

`WorldModelPolicy` (planning with a world model) comes with the planning solvers.

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Callable

import numpy as np

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.protocols import Actionable, Transformable

In [ ]:
#| export
class BasePolicy:
    """
    Base class of the policies a :class:`World` runs.

    Attributes
    ----------
    env : EnvPool
        The pool of envs the policy acts in (set by :meth:`set_env`)
    type : str
        Identifier of the policy type
    """
    env: Any
    type: str

    def __init__(self, **kwargs: Any):
        self.env = None
        self.type = 'base'
        for arg, value in kwargs.items():
            setattr(self, arg, value)

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        "Actions of shape (num_envs, num_agents) for the stacked `info_dict`."
        raise NotImplementedError

    def set_env(self, env: Any):
        "Attach the pool of envs this policy acts in."
        self.env = env

    def _prepare_info(self, info_dict: dict) -> dict:
        """
        A copy of `info_dict` ready for a torch model:

        * ``self.process[key]`` (a scaler with ``transform``, e.g. from sklearn) is applied to
          the features of `key`, i.e. its last axis
        * ``self.transform[key]`` (a callable on a batch of channel-first images) is applied to
          the images of `key`, of shape (..., H, W, C): it gets them as (N, C, H, W), and the
          result is reshaped back to (..., C', H', W')
        * numeric numpy arrays become torch tensors
        """
        import torch
        process, transform = getattr(self, 'process', {}), getattr(self, 'transform', {})
        out = {}
        for key, value in info_dict.items():
            if key in process:
                value = np.asarray(value)
                value = process[key].transform(value.reshape(-1, value.shape[-1])).reshape(value.shape)
            if key in transform:
                value = torch.as_tensor(value)
                images = value.reshape(-1, *value.shape[-3:]).permute(0, 3, 1, 2)
                images = transform[key](images)
                value = images.reshape(*value.shape[:-3], *images.shape[1:])
            if isinstance(value, np.ndarray) and value.dtype.kind not in 'USO':
                value = torch.from_numpy(value)
            out[key] = value
        return out

In [ ]:
#| export
class RandomPolicy(BasePolicy):
    "Samples uniformly random actions from the pool's batched action space."
    def __init__(self, seed: int | None = None, **kwargs: Any):
        """
        `seed` seeds the action space when the policy is attached to a pool,
        making the sampled actions reproducible.
        """
        super().__init__(**kwargs)
        self.type = 'random'
        self.seed = seed

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        return self.env.action_space.sample()

    def set_env(self, env: Any):
        super().set_env(env)
        if self.seed is not None:
            self._set_seed(self.seed)

    def _set_seed(self, seed: int):
        if self.env is not None:
            self.env.action_space.seed(seed)

In [ ]:
#| export
class ExpertPolicy(BasePolicy):
    """
    Base class of privileged (scripted) policies, which read the true state of each env
    instead of the info dict, e.g. a shortest-path planner to collect expert datasets.

    Subclasses implement :meth:`act`, the actions of all the agents of one env.
    """
    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self.type = 'expert'

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        return np.array([self.act(env.unwrapped) for env in self.env.envs])

    def act(self, env: MultiAgentEnv) -> np.ndarray:
        "Actions of the agents of `env`, of shape (num_agents,)."
        raise NotImplementedError

In [ ]:
#| export
class FeedForwardPolicy(BasePolicy):
    """
    Computes actions with one forward pass of a torch model, e.g. a behaviour-cloned policy or
    a trained MARL actor.

    ``model.get_action(info_dict)`` receives the prepared info dict (see
    :meth:`BasePolicy._prepare_info`), with its tensors on the model's device, and returns
    actions of shape (num_envs, num_agents). ``process['action']``, if given, maps them back
    with its ``inverse_transform``.
    """
    def __init__(
        self,
        model: Actionable,
        process: dict[str, Transformable] | None = None,
        transform: dict[str, Callable] | None = None,
        **kwargs: Any):
        if not isinstance(model, Actionable):
            raise TypeError(f"FeedForwardPolicy needs a model with `get_action` (Actionable), got {type(model).__name__}")
        super().__init__(**kwargs)
        self.type = 'feed_forward'
        self.model = model.eval()
        self.process = process or {}
        self.transform = transform or {}

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        import torch
        info = self._prepare_info(info_dict)
        parameter = next(self.model.parameters(), None)
        if parameter is not None:
            info = {k: v.to(parameter.device) if torch.is_tensor(v) else v for k, v in info.items()}
        with torch.no_grad():
            action = self.model.get_action(info)
        if torch.is_tensor(action):
            action = action.cpu().numpy()
        if 'action' in self.process:
            action = self.process['action'].inverse_transform(action)
        return action

Tested with the `World` in `world/01_world.ipynb`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()